In [ ]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# 关闭 Optuna 的逐次试验打印，保持终端输出清爽
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_Metrics.csv"
# 若修改后速度依然不理想，可将此处改为 15 或 20
OPTUNA_TRIALS = 30  
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_train_np, y_train_np):
    """
    加速版 Optuna 内层目标函数：
    使用 80/20 Hold-out 验证代替多折交叉验证，大幅减少评估单组超参数的时间。
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.2, 0.6),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 将当前的训练集按 80% 训练 / 20% 验证进行单次随机划分
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_np, y_train_np, test_size=0.2, random_state=42
    )
    
    model = XGBRFRegressor(**params)
    
    # 显式 GPU 训练
    model.fit(cp.array(X_tr), cp.array(y_tr))
    
    # 预测并计算 MSE
    preds = cp.asnumpy(model.predict(cp.array(X_val)))
    mse = mean_squared_error(y_val, preds)
        
    return mse

def run_nested_optimization(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"文件未找到: {input_file}。请先运行数据固化脚本 01_prepare_P4_data.py。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*70}")
    print(f">>> 启动嵌套 LOYO 交叉验证与超参数优化 (加速版)")
    print(f"    数据集特征维度: {len(feature_cols)}")
    print(f"    每个外层折叠的 Optuna 寻优次数: {OPTUNA_TRIALS}")
    print(f"{'='*70}")

    for test_year in years:
        # 1. 划分外层训练集与测试集
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_np = train_df[feature_cols].values
        y_train_np = train_df['yield'].values
        X_test_np = test_df[feature_cols].values
        y_test_np = test_df['yield'].values
        
        # 2. 内层优化：启动 Optuna
        print(f"    [Year {test_year}] 正在寻找年度最优参数...", end="", flush=True)
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
        
        func = lambda trial: objective(trial, X_train_np, y_train_np)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        best_params = study.best_params
        best_params.update({'tree_method': 'hist', 'device': 'cuda', 'random_state': 42, 'n_jobs': -1})
        print(f" 完成. (最佳深度: {best_params['max_depth']}, colsample: {best_params['colsample_bynode']:.2f})")
        
        # 3. 年度模型定型：使用最佳参数在整个年度训练集上拟合
        X_train_gpu = cp.array(X_train_np)
        y_train_gpu = cp.array(y_train_np)
        X_test_gpu = cp.array(X_test_np)
        
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(X_train_gpu, y_train_gpu)
        
        # 4. 最终评估：预测留出的测试年数据
        y_pred_gpu = cp.asarray(final_model.predict(X_test_gpu))
        y_pred = cp.asnumpy(y_pred_gpu)
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        # 记录单年结果
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"    -> [验证结果] R2: {m['R2']:.3f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 5. 计算全局拼接池化指标
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Best_Params'] = 'N/A (Nested)'
    fold_results.append(global_metrics)
    
    # 打印最终成果
    print(f"{'='*70}")
    print(f">>> 嵌套交叉验证全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*70}")

    # 导出至 CSV
    cols = ['Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(fold_results)[cols]
    results_df.to_csv(output_csv, index=False)
    print(f"学术评估指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_optimization(INPUT_FILE, OUTPUT_CSV)